<a href="https://colab.research.google.com/github/serelk/nebius-academy-course/blob/main/week3/task-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community

!pip install pypdf
!pip install -qU langchain-community faiss-cpu
!pip install -U langchain-huggingface

In [ ]:
from datasets import load_dataset
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from uuid import uuid4

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
embedding_dim = 384  # Dimension for 'BAAI/bge-small-en-v1.5'
index = faiss.IndexFlatL2(embedding_dim)

In [ ]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from datasets import load_dataset
import pandas as pd # Import pandas to use notna

# Re-defining dataset and filtered_df to ensure they are in scope
dataset = load_dataset("PatronusAI/financebench",)
df = dataset["train"].to_pandas()
filtered_df = df.where(df.question_type == "domain-relevant").dropna().sort_values(by="financebench_id").head(5)

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # A common chunk size
    chunk_overlap=50, # A common overlap
    length_function=len,
    is_separator_regex=False,
)

all_processed_docs = []

for index , row in filtered_df.head(1).iterrows():
  print(row.doc_name, row.company, row.doc_period)
  loader = PyPDFLoader(row.doc_link, mode="page")
  docs = loader.load() # docs here are full pages from the PDF

  for idx , doc in enumerate(docs):
    # Prepare metadata for this specific document (page)
    metadata = doc.metadata.copy()
    metadata['doc_name'] = row.doc_name
    metadata['company'] = row.company
    metadata['doc_period'] = row.doc_period
    # Safely add evidence_page_num if it exists
    if isinstance(row.evidence, list) and len(row.evidence) > 0 and "evidence_page_num" in row.evidence[0]:
        metadata['evidence_page_num'] = row.evidence[0]["evidence_page_num"]
    metadata['page_number'] = idx

    # Split the document content (page_content) into smaller chunks
    # and assign the prepared metadata to each chunk
    split_chunks = text_splitter.create_documents(
        texts=[doc.page_content],
        metadatas=[metadata] # Pass the prepared metadata to each chunk
    )
    all_processed_docs.extend(split_chunks)

# Generate UUIDs for all the split document chunks
uuids = [str(uuid4()) for _ in range(len(all_processed_docs))]
vector_store.add_documents(documents=all_processed_docs, ids=uuids)



In [ ]:
# Save the vector store locally
saved_path = "faiss_index"
vector_store.save_local(saved_path)
print(f"FAISS index saved to {saved_path}/")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After mounting your Google Drive, you can save the FAISS index to a specific folder within your Drive. Make sure the folder `faiss_indexes` exists, or create it.

In [ ]:
drive_path = '/content/drive/MyDrive/faiss_indexes'
import os
os.makedirs(drive_path, exist_ok=True)

drive_saved_path = os.path.join(drive_path, "my_faiss_index")
vector_store.save_local(drive_saved_path)
print(f"FAISS index saved to {drive_saved_path}/")

In [ ]:
!ls /content/drive/MyDrive/faiss_indexes/my_faiss_index/

Now, to use this index in another notebook, you would first mount your Google Drive as shown above, and then load the index using `FAISS.load_local`. Here's how you'd load it:

In [ ]:
# Load the vector store from Google Drive
loaded_vector_store = FAISS.load_local(
    drive_saved_path, embeddings, allow_dangerous_deserialization=True
)
print(f"FAISS index loaded from {drive_saved_path}/")

# You can then perform similarity searches with the loaded vector store
# For example:
# query_results = loaded_vector_store.similarity_search("Your query here", k=2)
# for res in query_results:
#     print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
results = loaded_vector_store.similarity_search(
    "Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.",
    k=2,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
results = loaded_vector_store.similarity_search(
    "Is Verizon a capital intensive business based on FY 2022 data?",
    k=2,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")